> **October 2026 Colab fix:** Cell 7 now renders the chat template as text, tokenizes it once, moves `input_ids` and `attention_mask` tensors to CUDA separately, and passes both to `model.generate`. This handles modern Transformers `BatchEncoding` objects and records nonfatal chat errors instead of preventing the final results ZIP.

> **Dataset loader fix (October 2026):** WikiText-2 is loaded from the explicit Hugging Face repository `Salesforce/wikitext`, configuration `wikitext-2-raw-v1`. This fixes `HfUriError: Repository id must be 'namespace/name'` on current Colab images. No library downgrades are required.

# SmolLM2-360M V10: selective CeNN {−1,0,+1} replacements — real quality + memory test

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/SmolLM2_360M_Ternary_CeNN_Selective_Replacement_V10_Colab.ipynb)

**This tests the actual decoder:** [HuggingFaceTB/SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct), not synthetic random matrices.

**Evidence from uploaded Laya V9 T4 ZIP:** 86.6–86.7% saved per ternary projection; 11% relative reconstruction MSE; QKV 512-row ternary 11.06ms vs native FP16 0.191ms; CeNN branch concat added 18 MiB at B4/T512/D768 and separate QKV concat added 9 MiB. Therefore the Smol experiment **never concatenates branches or claims native ternary acceleration**.

**Controlled architectures:** (A) replace selected late MLP-down and attention-output linears with strict 2-bit ternary G128 projections; (B) optionally add rank-16 FP16 residual recovery (not strictly ternary as a complete layer); (C) test replacing one attention-output projection by a **pure CeNN 3-tap −1/0/+1 stencil**, as a deliberately risky negative control.

**Tests:** disjoint WikiText train-development pools for selection/guard; validation decision agreement with original FP16 model, gold WikiText validation + test perplexity, token next-word accuracy, real instruction chat, FP16 vs ternary model memory and latency. No training/fine-tuning, no custom GPU compilation. Accept zero replacements if quality fails. Export ZIP + exact-source-SHA packed delta, with reload parity.


**Causality requirement:** The CeNN-only attention-output negative control uses only left/current token taps (right coefficient zero). It is evaluated on full-prefix teacher comparisons, **not deployed for cached autoregressive decoding**, because it does not implement cross-token recurrence state for KV caching.


In [ ]:
#@title 1. Install only missing libraries; no custom Triton build
import os,sys,json,gc,math,random,zipfile,copy,subprocess,importlib.util
from pathlib import Path
from urllib.request import urlopen
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
needed=[(x,s) for x,s in [
 ("transformers","transformers>=4.50,<5"),
 ("datasets","datasets>=2.20,<5"),
 ("safetensors","safetensors>=0.4"),
 ("huggingface_hub","huggingface_hub>=0.35"),
 ("pandas","pandas")]
 if importlib.util.find_spec(x) is None]
if needed:
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "--only-binary=:all:",*[spec for _,spec in needed]],check=True)
from transformers import AutoModelForCausalLM,AutoTokenizer
from huggingface_hub import HfApi
from safetensors.torch import save_file,load_file
assert torch.cuda.is_available(),"Enable T4/L4 GPU in Colab"
print("GPU:",torch.cuda.get_device_name(0),"PyTorch",torch.__version__)


In [ ]:
#@title 2. Research controls, revision-pinned codec and pretrained SmolLM2
SOURCE="HuggingFaceTB/SmolLM2-360M-Instruct" #@param {type:"string"}
LAST_LAYERS=4 #@param {type:"integer"}
MAX_TRIALS=6 #@param {type:"integer"}
MAX_ACCEPTED=2 #@param {type:"integer"}
TERNARY_TILE=128 #@param {type:"integer"}
RANK16_RESCUE=True #@param {type:"boolean"}
MAX_NEW_ERRORS=0 #@param {type:"integer"}
MAX_MEAN_JS=0.016 #@param {type:"number"}
DEV_WINDOWS=8 #@param {type:"integer"}
GUARD_WINDOWS=8 #@param {type:"integer"}
AUDIT_WINDOWS=12 #@param {type:"integer"}
TEST_WINDOWS=12 #@param {type:"integer"}
SEQ_LEN=128 #@param {type:"integer"}
RUN_CENN_ONLY_CONTROL=True #@param {type:"boolean"}
RUN_CHAT=True #@param {type:"boolean"}
CHAT_MAX_NEW_TOKENS=32 #@param {type:"integer"}
OUT=Path("/content/SmolLM2_360M_Ternary_CeNN_V10")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(2069);np.random.seed(2069);torch.manual_seed(2069)
assert DEV_WINDOWS>0 and GUARD_WINDOWS>0 and 0<=MAX_ACCEPTED<=MAX_TRIALS
try:
 from google.colab import userdata
 TOKEN=userdata.get("HF_TOKEN")
except Exception:TOKEN=os.environ.get("HF_TOKEN")
REV=HfApi(token=TOKEN).model_info(SOURCE).sha
def import_remote(sha,filename,name):
    path=Path("/content")/filename
    with urlopen(f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{sha}/src/tinycenn_lm/{filename}",timeout=90) as f:
        path.write_bytes(f.read())
    spec=importlib.util.spec_from_file_location(name,str(path))
    mod=importlib.util.module_from_spec(spec)
    sys.modules[name]=mod;spec.loader.exec_module(mod)
    return mod
V9_REV="4ea3bf6e5001b7b123d7c137cb09f0ce32cc1c55"
V10_REV="6a72d184349b735da6ae01c75452f30f10624579"
v9=import_remote(V9_REV,"laya_cenn_ternary_concat_v9.py",
                 "laya_cenn_ternary_concat_v9")
ternary=import_remote(V10_REV,"smol_ternary_cenn_v10.py",
                      "smol_ternary_cenn_v10")
tokenizer=AutoTokenizer.from_pretrained(SOURCE,revision=REV,token=TOKEN)
if tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token
teacher=AutoModelForCausalLM.from_pretrained(
    SOURCE,revision=REV,token=TOKEN,torch_dtype=torch.float16,
    attn_implementation="sdpa").cuda().eval().requires_grad_(False)
names=ternary.candidate_names(teacher,last_layers=LAST_LAYERS)
assert names and len(teacher.model.layers)>0
print("Source SHA",REV,"layers",len(teacher.model.layers),
      "candidate sites",names,
      "FP16 state MiB",round(ternary.model_storage_bytes(teacher)/2**20,1))
# Random-matrix numerical smoke test for packed ternary and rank16 recovery.
probe=nn.Linear(256,128,device="cuda",dtype=torch.float16)
plain=ternary.SmolTernaryProjection.from_linear(probe,chunk_rows=64,rank=0)
rescue=ternary.SmolTernaryProjection.from_linear(probe,chunk_rows=64,rank=16)
x=torch.randn(2,7,256,device="cuda",dtype=torch.float16)
assert plain(x).shape==rescue(x).shape==probe(x).shape
print("Packed projection smoke test PASS")
del probe,plain,rescue,x
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 3. Genuine WikiText windows: disjoint train selection/guard; untouched validation/test
from datasets import load_dataset

# Hugging Face Hub now requires a fully namespaced dataset repo ID.
# "wikitext" alone triggers HfUriError in recent datasets/huggingface_hub.
WIKITEXT_REPO="Salesforce/wikitext"
WIKITEXT_CONFIG="wikitext-2-raw-v1"
wiki=load_dataset(WIKITEXT_REPO,WIKITEXT_CONFIG)
assert all(split in wiki for split in ("train","validation","test")), (
    "WikiText-2 must provide train, validation and test splits.")
assert all("text" in wiki[split].column_names for split in ("train","validation","test"))
print("WikiText-2 dataset loaded:",WIKITEXT_REPO,WIKITEXT_CONFIG,
      {split:len(wiki[split]) for split in ("train","validation","test")})
def windowize(records,n,L=SEQ_LEN):
    ids=[]
    for row in records:
        text=row["text"].strip()
        if len(text)<70:continue
        ids+=tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id]
        if len(ids)>=(n+1)*(L+1):break
    result=[ids[i*(L+1):(i+1)*(L+1)] for i in range(n)]
    if any(len(row)!=L+1 for row in result):
        raise ValueError("Not enough text: increase train rows or lower WINDOWS")
    return result
train=wiki["train"]
dev=windowize(train.select(range(min(600,len(train)))),DEV_WINDOWS)
guard=windowize(train.select(range(600,min(1500,len(train)))),GUARD_WINDOWS)
audit=windowize(wiki["validation"],AUDIT_WINDOWS)
test=windowize(wiki["test"],TEST_WINDOWS)
def to_batches(windows,size=2):
    chunks=[]
    for i in range(0,len(windows),size):
        ids=torch.tensor([w[:-1] for w in windows[i:i+size]],
                         dtype=torch.long,device="cuda")
        chunks.append({"input_ids":ids,"attention_mask":torch.ones_like(ids)})
    return chunks
DEV=to_batches(dev);GUARD=to_batches(guard);AUDIT=to_batches(audit)
@torch.inference_mode()
def references(batches):
    return [teacher(**batch,use_cache=False).logits[:,-1,:].float().cpu()
            for batch in batches]
REF_DEV=references(DEV);REF_GUARD=references(GUARD)
print("Train DEV",len(dev),"GUARD",len(guard),
      "| independent validation",len(audit),"test",len(test))


In [ ]:
#@title 4. Pure CeNN-only projection replacement (negative control; restore teacher)
CENN_ONLY=None
if RUN_CENN_ONLY_CONTROL:
    target=next(n for n in reversed(names) if n.endswith("self_attn.o_proj"))
    old=ternary.replace(teacher,target,
             ternary.CeNNOnlyProjection.from_linear(teacher.get_submodule(target)))
    try:
        score=ternary.teacher_metrics(teacher,REF_DEV,DEV)
        CENN_ONLY=dict(module=target,**ternary.public(score),
          state_saving_bytes=ternary.projection_storage_bytes(old)-
                             ternary.projection_storage_bytes(teacher.get_submodule(target)))
        print("Untrained pure CeNN-only next-token result",CENN_ONLY)
    finally:ternary.replace(teacher,target,old)
    (OUT/"pure_cenn_replacement_control.json").write_text(
        json.dumps(CENN_ONLY,indent=2))
assert ternary.teacher_metrics(teacher,REF_DEV,DEV)["errors"]==0


In [ ]:
#@title 5. Search selective SmolLM2 ternary substitutes on train data ONLY
student=copy.deepcopy(teacher).eval().requires_grad_(False)
FP16_BYTES=ternary.model_storage_bytes(teacher)
BASE_DEV=ternary.teacher_metrics(student,REF_DEV,DEV)
BASE_GUARD=ternary.teacher_metrics(student,REF_GUARD,GUARD)
CURRENT_DEV,CURRENT_GUARD=BASE_DEV,BASE_GUARD
selected=[];trials=[]
for name in names[:MAX_TRIALS]:
    if len(selected)>=MAX_ACCEPTED:break
    old=student.get_submodule(name)
    options=[("strict_ternary",0)]
    if RANK16_RESCUE:options.append(("ternary_plus_rank16",16))
    for style,rank in options:
        entry=dict(module=name,style=style,rank=rank,accepted=False)
        try:
            candidate=ternary.SmolTernaryProjection.from_linear(
                old,chunk_rows=TERNARY_TILE,rank=rank)
            ternary.replace(student,name,candidate)
            d=ternary.teacher_metrics(student,REF_DEV,DEV)
            g=ternary.teacher_metrics(student,REF_GUARD,GUARD)
            keep=(ternary.acceptance(d,CURRENT_DEV,BASE_DEV,
                        max_new_errors=MAX_NEW_ERRORS,max_js=MAX_MEAN_JS)
                   and ternary.acceptance(g,CURRENT_GUARD,BASE_GUARD,
                        max_new_errors=MAX_NEW_ERRORS,max_js=MAX_MEAN_JS))
            entry.update(accepted=bool(keep),dev=ternary.public(d),
               guard=ternary.public(g),
               old_bytes=ternary.projection_storage_bytes(old),
               compressed_bytes=ternary.projection_storage_bytes(candidate))
            if keep:
                selected.append({"name":name,"rank":rank,"style":style,
                                 "tile_rows":TERNARY_TILE})
                CURRENT_DEV,CURRENT_GUARD=d,g
                print("KEEP",name,style,"teacher agreement",d["teacher_agreement"],
                      "guard",g["teacher_agreement"])
            else:ternary.replace(student,name,old)
        except Exception as ex:
            ternary.replace(student,name,old)
            entry["error"]=repr(ex)[:500]
            keep=False
            print("REJECT",name,style,repr(ex)[:180])
        trials.append(entry)
        if keep:break
    (OUT/"selection_progress.json").write_text(json.dumps(
        {"chosen":selected,"trials":trials},indent=2))
print("Selected",len(selected),"of",len(names),
      "| persistent state MiB",ternary.model_storage_bytes(student)/2**20)


In [ ]:
#@title 6. Independent teacher agreement, gold perplexity, and next-token accuracy
@torch.inference_mode()
def gold_metrics(model,windows):
    nll=0.;N=0;correct=0;last_count=0
    for i in range(0,len(windows),2):
        block=torch.tensor(windows[i:i+2],device="cuda")
        logits=model(block[:,:-1],use_cache=False).logits.float()
        truth=block[:,1:]
        nll+=float(F.cross_entropy(logits.reshape(-1,logits.shape[-1]),
                   truth.reshape(-1),reduction="sum"))
        N+=truth.numel()
        correct+=int((logits[:,-1,:].argmax(-1)==truth[:,-1]).sum())
        last_count+=len(truth)
    return {"tokens":N,"ppl":math.exp(min(50,nll/N)),
            "next_gold_accuracy":correct/last_count}
# Freeze choices BEFORE touching validation or test.
REF_AUDIT=references(AUDIT)
AUDIT_SCORE=ternary.teacher_metrics(student,REF_AUDIT,AUDIT)
GOLD={label:{"validation":gold_metrics(m,audit),"test":gold_metrics(m,test)}
      for label,m in (("original_fp16",teacher),("ternary_candidate",student))}
print("Independent validation agreement:",ternary.public(AUDIT_SCORE))
print("Gold label PPL and next-token accuracy:",json.dumps(GOLD,indent=2))
(OUT/"independent_quality.json").write_text(json.dumps({
    "dev":ternary.public(CURRENT_DEV),
    "guard":ternary.public(CURRENT_GUARD),
    "audit":ternary.public(AUDIT_SCORE),"gold":GOLD,
    "pure_cenn_control":CENN_ONLY},indent=2))


In [ ]:
#@title 7. End-to-end latency / memory and deterministic chat examples
LATENCY=[];CHATS=[]
if selected:
    for B in (1,4):
        for T in (64,128):
            ids=torch.randint(0,tokenizer.vocab_size,(B,T),
                              device="cuda")
            b={"input_ids":ids,"attention_mask":torch.ones_like(ids)}
            perf=ternary.benchmark_pair(teacher,student,[b],repeats=4)[0]
            for label,model in (("original",teacher),("candidate",student)):
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
                begin=torch.cuda.memory_allocated()
                out=model(**b,use_cache=False).logits
                torch.cuda.synchronize()
                perf[label+"_incremental_peak_mib"]=(
                    torch.cuda.max_memory_allocated()-begin)/2**20
                del out
            LATENCY.append(perf)
    display(pd.DataFrame(LATENCY))
else:
    print("Zero ternary modules qualified; no false native speedup benchmark.")
pd.DataFrame(LATENCY).to_csv(OUT/"end_to_end_latency.csv",index=False)
if RUN_CHAT:
    prompts=[
     "Explain why the sky looks blue in two sentences.",
     "What is 17 multiplied by 23?"]
    for question in prompts:
        row={"question":question}
        try:
            # Modern Transformers can return BatchEncoding (not a Tensor)
            # from apply_chat_template(..., tokenize=True). Never call .cuda()
            # on BatchEncoding: encode the rendered text and move its tensors.
            prompt_text=tokenizer.apply_chat_template(
                [{"role":"user","content":question}],
                add_generation_prompt=True,tokenize=False)
            encoded=tokenizer(
                prompt_text,add_special_tokens=False,return_tensors="pt")
            input_ids=encoded["input_ids"].to("cuda")
            attention_mask=encoded.get("attention_mask")
            if attention_mask is None:
                attention_mask=torch.ones_like(encoded["input_ids"])
            attention_mask=attention_mask.to("cuda")
            assert input_ids.ndim==2 and attention_mask.shape==input_ids.shape
            for label,model in (("original",teacher),("ternary",student)):
                with torch.inference_mode():
                    reply=model.generate(
                        input_ids=input_ids,attention_mask=attention_mask,
                        max_new_tokens=CHAT_MAX_NEW_TOKENS,do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id)
                row[label]=tokenizer.decode(
                    reply[0,input_ids.shape[1]:],
                    skip_special_tokens=True)
            print("\\nQuestion:",question,"\\nFP16:",row["original"],
                  "\\nTernary:",row["ternary"])
        except Exception as error:
            # Chat demonstration is optional; preserving scientific results
            # and final statistics archive matters more than a failed prompt.
            row["generation_error"]=repr(error)[:600]
            print("CHAT COMPARISON FAILED (nonfatal):",row["generation_error"])
        CHATS.append(row)
(OUT/"chat_generation_comparison.json").write_text(json.dumps(CHATS,indent=2))


In [ ]:
#@title 8. Export exact packed ternary delta, verify reload, and download all evidence
manifest={
 "base_repo":SOURCE,"base_sha":REV,
 "evaluation_dataset":WIKITEXT_REPO,
 "evaluation_dataset_config":WIKITEXT_CONFIG,
 "codec_rev":V9_REV,"runtime_rev":V10_REV,
 "replacements":selected,
 "format":"signed {-1,0,+1} 2-bit G128 with optional rank16 FP16 rescue",
 "requires_original_fp16_model":True,
 "not_native_ternary_gemm":True}
(OUT/"delta_manifest.json").write_text(json.dumps(manifest,indent=2))
DELTA=OUT/"smollm2_v10_ternary_delta.safetensors"
if selected:
    tensors={}
    for entry in selected:
        for key,t in student.get_submodule(entry["name"]).state_dict().items():
            tensors[entry["name"]+"."+key]=t.detach().cpu().contiguous()
    save_file(tensors,str(DELTA))
    # Cold load the exact packed delta without rerunning ternarization/SVD.
    test_model=copy.deepcopy(teacher).eval().requires_grad_(False)
    recovered=load_file(str(DELTA),device="cpu")
    for entry in selected:
        name=entry["name"];old=test_model.get_submodule(name)
        base=v9.PackedTernaryLinear(
            old.in_features,old.out_features,old.bias is not None,
            entry["tile_rows"],"prealloc",device="cuda")
        module=ternary.SmolTernaryProjection(base,rank=entry["rank"])
        state={key[len(name)+1:]:value.cuda()
               for key,value in recovered.items() if key.startswith(name+".")}
        if entry["rank"]>0:
            module.residual_left=torch.empty_like(state["residual_left"])
            module.residual_right=torch.empty_like(state["residual_right"])
        module.load_state_dict(state,strict=True)
        ternary.replace(test_model,name,module)
    with torch.inference_mode():
        a=student(**DEV[0],use_cache=False).logits[:,-1]
        b=test_model(**DEV[0],use_cache=False).logits[:,-1]
        torch.testing.assert_close(a,b,atol=.004,rtol=.004)
    print("Delta cold-reload parity PASS | MiB",DELTA.stat().st_size/2**20)
    del test_model,recovered,tensors,a,b
    gc.collect();torch.cuda.empty_cache()
else:print("No accepted modules; no compressed checkpoint falsely exported.")
final={
 "source":SOURCE,"sha":REV,"gpu":torch.cuda.get_device_name(0),
 "selected":selected,"trials":trials,
 "fp16_model_mib":FP16_BYTES/2**20,
 "candidate_model_mib":ternary.model_storage_bytes(student)/2**20,
 "model_state_saving_percent":100*(1-ternary.model_storage_bytes(student)/FP16_BYTES),
 "independent_validation_agreement":ternary.public(AUDIT_SCORE),
 "gold_ppl":GOLD,"latency":LATENCY,"chat":CHATS,
 "pure_cenn_only_control":CENN_ONLY,
 "production_validated":False,
 "caveats":[
    "2-bit weight state, not native low-bit CUDA GEMM",
    "No concatenation: V9 measured extra peak activation allocations",
    "Rank16 residual makes full replacement not strictly ternary",
    "Gold perplexity is independent of teacher decision agreement",
    "FP16 tile decoding may greatly slow inference"]}
(OUT/"summary_smol_ternary_v10.json").write_text(json.dumps(final,indent=2))
OUTZIP=OUT/"SmolLM2_360M_Ternary_CeNN_V10_Results.zip"
with zipfile.ZipFile(OUTZIP,"w",zipfile.ZIP_DEFLATED) as zipf:
    for f in OUT.iterdir():
        if f.is_file() and f!=OUTZIP:zipf.write(f,f.name)
print("Results:",OUTZIP)
try:
    from google.colab import files
    files.download(str(OUTZIP))
except ImportError:pass


## Interpret actual SmolLM results

If no ternary projections pass both source-case-disjoint development gates, a **zero-replacement result** is correct. The pure CeNN-only negative control will show how replacing learned attention outputs with an untrained local {-1,0,+1} stencil affects decisions.

**Successful storage compression is not inference acceleration:** G128 2-bit packed weights decode FP16 tiles during inference. The V9 memory results already showed this is often much slower than an optimized FP16 matrix multiply.

Only call the replacement useful when independent WikiText validation/test perplexity and chat outputs remain strong while model state actually shrinks. Even then, production readiness requires broader evaluation and a faster native ternary kernel.
